# 📊 EDA + Entraînement + Benchmark complet
## Reconnaissance du genre - Noms propres ivoiriens | SIGAP / AFOR
**INP-HB · ESI · Stage 2A · 2025-2026**

Ce notebook fait **tout en un** :
1. Analyse exploratoire complète du corpus (EDA)
2. Entraînement des **8 modèles classiques** (Naive Bayes, SVM, Random Forest…)
3. Entraînement du **CNN + BiLSTM + MultiHeadAttention v4** (TensorFlow)
4. Comparaison finale sur toutes les métriques
5. Visualisations pour le mémoire (ROC, matrices de confusion, radar…)

> 📁 **Seul fichier requis :** `dataset_nettoye.csv` - même dossier que ce notebook


## 0. Installation et imports

In [ ]:
# Installation des dépendances (exécuter une seule fois)
import subprocess, sys

pkgs = ['tensorflow', 'scikit-learn', 'pandas', 'numpy',
        'matplotlib', 'seaborn']
for pkg in pkgs:
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)
print("✅ Dépendances OK")


In [ ]:
import os, re, time, warnings, unicodedata, pickle, random
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'   # silence TF logs
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
import seaborn as sns
from collections import Counter

# ── Scikit-learn ────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, precision_score, recall_score,
    classification_report, confusion_matrix, roc_curve, auc,
    average_precision_score, matthews_corrcoef,
    precision_recall_curve, ConfusionMatrixDisplay
)

# ── TensorFlow / Keras ──────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (
    Embedding, Bidirectional, LSTM, Dense, Dropout,
    SpatialDropout1D, Conv1D,
    MultiHeadAttention, LayerNormalization, Add
)
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

# ── Palette ─────────────────────────────────────────────────────────────
ORANGE = '#F3802F'; BLUE = '#001F5F'; LBLUE = '#2E75B6'
GREEN  = '#009E5F'; PINK = '#E63F7A'; GRAY  = '#8E9BB5'
MODEL_COLORS = {
    'Naive Bayes (Multinomial)': '#A8DADC',
    'Naive Bayes (Complement)':  '#76B7B2',
    'Régression Logistique':     '#59A14F',
    'SGD Classifier':            '#8CD17D',
    'SVM Linéaire':              '#F28E2B',
    'Random Forest':             '#4E79A7',
    'Gradient Boosting':         '#B6992D',
    'Arbre de Décision':         '#BAB0AC',
    'CNN + BiLSTM + MHA v4':     '#E15759',
}
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi':130,'figure.facecolor':'white',
                     'axes.spines.top':False,'axes.spines.right':False})
os.makedirs('figures', exist_ok=True)
print(f" TensorFlow {tf.__version__} | Environnement prêt")


## 1. Chargement et prétraitement

In [ ]:
# ── Hyperparamètres ────────────────────
DATASET_PATH     = 'dataset_nettoye.csv'
N_AUGMENTATIONS  = 5
WORD_DROP_PROB   = 0.30
PRENOMS_FIRST    = 0.50
CHAR_NOISE_PROB  = 0.15
EMBED_DIM        = 96
LSTM1            = 128
LSTM2            = 64
MHA_HEADS        = 4
MHA_KEY_DIM      = 32
CNN_FILTERS      = 128
BATCH_SIZE       = 128
EPOCHS           = 20
LABEL_SMOOTH     = 0.08

def clean_text(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    text = unicodedata.normalize('NFKD', text).encode('ASCII','ignore').decode('ASCII')
    text = re.sub(r"['-]", ' ', text)
    text = re.sub(r'[^a-z\s]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

df = pd.read_csv(DATASET_PATH)
df['nom_clean']     = df['NOM'].apply(clean_text)
df['prenoms_clean'] = df['PRENOMS'].fillna('').apply(clean_text)
df['full_name']     = (df['nom_clean'] + ' ' + df['prenoms_clean']).str.strip()
df = df[df['full_name'] != ''].reset_index(drop=True)
df['label']    = (df['GENRE'] == 'M').astype(int)
df['full_len'] = df['full_name'].str.len()
df['nb_tokens']= df['full_name'].str.split().str.len()

# Découpage FIGÉ (split_reference.csv), commun à tous les notebooks : le recalculer après
# une modification du dataset déplacerait des noms entre train, validation et test
import hashlib
sp  = pd.read_csv('split_reference.csv')
emp = df['full_name'].map(lambda s: hashlib.sha1(s.encode()).hexdigest()[:10]).values
assert len(sp) == len(df) and (sp['empreinte'].values == emp).all(),     "split_reference.csv ne correspond pas au dataset"
ens = sp['ensemble'].values
X_train, X_val, X_test = (df.loc[ens == e, 'full_name'] for e in ('train', 'val', 'test'))
y_train, y_val, y_test = (df.loc[ens == e, 'label'] for e in ('train', 'val', 'test'))

y_test_arr = y_test.values
print(f"Dataset  : {len(df):,} entrées  |  M={df.label.sum():,} ({df.label.mean()*100:.1f}%)  F={(1-df.label).sum():,} ({(1-df.label).mean()*100:.1f}%)")
print(f"Train    : {len(X_train):,}  |  Val : {len(X_val):,}  |  Test : {len(X_test):,}")
df[['NOM','PRENOMS','GENRE','full_name']].head(8)


## 2. Analyse Exploratoire (EDA)

In [ ]:
# ── Fig 1 : Distribution des genres ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Distribution des genres - corpus de noms ivoiriens", fontsize=14, fontweight='bold', color=BLUE)
counts = df['GENRE'].value_counts()
axes[0].pie(counts.values, labels=['Masculin','Féminin'], colors=[LBLUE,PINK],
            autopct='%1.1f%%', startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title('Répartition M / F', color=BLUE)
bars = axes[1].bar(['Masculin','Féminin'], counts.values, color=[LBLUE,PINK], edgecolor='white', width=0.5)
for b,v in zip(bars, counts.values):
    axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+80,
                 f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')
axes[1].set_ylim(0, counts.max()*1.2)
axes[1].set_ylabel('Effectif')
plt.tight_layout(); plt.savefig('figures/fig1_genres.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Ratio M/F = {counts['M']/counts['F']:.3f}")


In [ ]:
# ── Fig 2 : Longueurs de séquence ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Longueurs des séquences (full_name après clean_text)", fontsize=13, fontweight='bold', color=BLUE)

for g, c in [('M', LBLUE), ('F', PINK)]:
    axes[0].hist(df[df.GENRE==g]['full_len'], bins=40, alpha=0.6, color=c, label=g, edgecolor='white')
for pct, ls in [(50,'--'),(90,':'),(99,'-.')]:
    v = df['full_len'].quantile(pct/100)
    axes[0].axvline(v, color=ORANGE, ls=ls, lw=2, label=f'P{pct}={v:.0f}')
axes[0].set_title('Distribution longueurs'); axes[0].legend(fontsize=9)

data_bp = [df[df.GENRE=='M']['full_len'].values, df[df.GENRE=='F']['full_len'].values]
bp = axes[1].boxplot(data_bp, patch_artist=True, notch=True, medianprops=dict(color='white',lw=2))
bp['boxes'][0].set_facecolor(LBLUE); bp['boxes'][1].set_facecolor(PINK)
axes[1].set_xticklabels(['Masculin','Féminin']); axes[1].set_title('Boxplot par genre')

tok = df['nb_tokens'].value_counts().sort_index()
b = axes[2].bar(tok.index.astype(str), tok.values, color=LBLUE, edgecolor='white')
[axes[2].text(bb.get_x()+bb.get_width()/2, bb.get_height()+30, f'{v:,}', ha='center', fontsize=8) for bb,v in zip(b,tok.values)]
axes[2].set_title('Nb tokens par full_name'); axes[2].set_xlabel('Tokens')

plt.tight_layout(); plt.savefig('figures/fig2_longueurs.png', dpi=150, bbox_inches='tight')
plt.show()
for p in [50,75,90,95,99]: print(f"P{p} = {df['full_len'].quantile(p/100):.0f} caractères")
print(f"→ max_len = 37 (P99) retenu dans le modèle")


In [ ]:
# ── Fig 3 : Prénoms discriminants ───────────────────────────────────
dfF, dfM = df[df.GENRE=='F'], df[df.GENRE=='M']
cnt_f = Counter([w for p in dfF['prenoms_clean'] for w in p.split() if w])
cnt_m = Counter([w for p in dfM['prenoms_clean'] for w in p.split() if w])
recs = []
for w in set(cnt_f)|set(cnt_m):
    f,m = cnt_f.get(w,0), cnt_m.get(w,0)
    tot = f+m
    if tot >= 25: recs.append({'mot':w,'freq':tot,'pct_f':f/tot})
wd = pd.DataFrame(recs)
top_f = wd[wd.pct_f>0.85].nlargest(14,'freq').sort_values('freq')
top_m = wd[wd.pct_f<0.15].nlargest(14,'freq').sort_values('freq')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Prénoms fortement discriminants (fréq. ≥ 25)", fontsize=13, fontweight='bold', color=BLUE)
for ax, data, color, title in [(axes[0],top_f,PINK,'♀ Très féminins (>85%F)'),
                                (axes[1],top_m,LBLUE,'♂ Très masculins (<15%F)')]:
    bars = ax.barh(data['mot'], data['freq'], color=color, edgecolor='white', alpha=0.9)
    for b,v,p in zip(bars,data['freq'],data['pct_f']):
        ps = p if color==PINK else 1-p
        ax.text(b.get_width()+3, b.get_y()+b.get_height()/2, f'{v} ({ps*100:.0f}%)', va='center', fontsize=9)
    ax.set_title(title, color=color, fontsize=12)
    ax.set_xlabel('Occurrences dans les prénoms')
    ax.set_xlim(0, data['freq'].max()*1.28)
plt.tight_layout(); plt.savefig('figures/fig3_prenoms.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Fig 4 : N-grammes discriminants ─────────────────────────────────
def get_ngrams(text, n):
    return [text[i:i+n] for i in range(len(text)-n+1)]

cnt_f3 = Counter([ng for t in df[df.GENRE=='F']['full_name'] for ng in get_ngrams(t,3)])
cnt_m3 = Counter([ng for t in df[df.GENRE=='M']['full_name'] for ng in get_ngrams(t,3)])
recs3 = []
for g in set(cnt_f3)|set(cnt_m3):
    f,m = cnt_f3.get(g,0),cnt_m3.get(g,0); tot=f+m
    if tot>=200: recs3.append({'ngram':g,'freq':tot,'pct_f':f/tot})
ng3 = pd.DataFrame(recs3)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
fig.suptitle("Trigrammes de caractères - fondement des Conv1D(k=3)", fontsize=13, fontweight='bold', color=BLUE)
for ax, data, color, title in [
    (axes[0], ng3[ng3.pct_f>0.82].nlargest(12,'freq').sort_values('freq'), PINK, '♀ Féminins (>82%F)'),
    (axes[1], ng3[ng3.pct_f<0.22].nlargest(12,'freq').sort_values('freq'), LBLUE,'♂ Masculins (<22%F)'),
]:
    bars = ax.barh(data['ngram'], data['freq'], color=color, edgecolor='white', alpha=0.9)
    for b,v,p in zip(bars,data['freq'],data['pct_f']):
        ps = p if color==PINK else 1-p
        ax.text(b.get_width()+5, b.get_y()+b.get_height()/2, f'{v} ({ps*100:.0f}%)', va='center', fontsize=9)
    ax.set_title(title, color=color)
    ax.set_xlabel('Occurrences'); ax.set_xlim(0, data['freq'].max()*1.28)
    ax.set_yticklabels([f'«{s}»' for s in data['ngram']], fontfamily='monospace')
plt.tight_layout(); plt.savefig('figures/fig4_ngrams.png', dpi=150, bbox_inches='tight')
plt.show()
print("→ «ata»=féminin (Fatoumata, Aminata) | «uel»=masculin (Emmanuel, Samuel)")


## 3. Entraînement des modèles classiques (ML)

In [ ]:
# Définition des 8 pipelines TF-IDF + classificateur
MODELS = {
    'Naive Bayes (Multinomial)': Pipeline([
        ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4), max_features=50000)),
        ('clf',   MultinomialNB())]),
    'Naive Bayes (Complement)': Pipeline([
        ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4), max_features=50000)),
        ('clf',   ComplementNB())]),
    'Régression Logistique': Pipeline([
        ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(2,5), max_features=100000)),
        ('clf',   LogisticRegression(C=1.0, max_iter=300, class_weight='balanced', random_state=42))]),
    'SGD Classifier': Pipeline([
        ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4), max_features=50000)),
        ('clf',   SGDClassifier(loss='modified_huber', alpha=1e-4, max_iter=100,
                                class_weight='balanced', random_state=42))]),
    'Random Forest': Pipeline([
        ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4), max_features=20000)),
        ('clf',   RandomForestClassifier(n_estimators=200, max_depth=20,
                                          class_weight='balanced', random_state=42, n_jobs=-1))]),
    'Gradient Boosting': Pipeline([
        ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4), max_features=20000)),
        ('clf',   GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42))]),
    'Arbre de Décision': Pipeline([
        ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(2,3), max_features=10000)),
        ('clf',   DecisionTreeClassifier(max_depth=20, class_weight='balanced', random_state=42))]),
}

RESULTS = {}
print(f"{'Modèle':<32} {'Acc':>7} {'F1':>7} {'AUC':>7} {'MCC':>7}  Temps")
print("─"*70)

for name, pipe in MODELS.items():
    t0 = time.time()
    pipe.fit(X_train, y_train)
    t_train = time.time() - t0

    t1 = time.time()
    y_pred = pipe.predict(X_test)
    t_inf  = (time.time()-t1)/len(X_test)*1000

    try:    y_prob = pipe.predict_proba(X_test)[:,1]
    except: y_prob = (pipe.decision_function(X_test) - pipe.decision_function(X_test).min())
    try:    y_prob /= y_prob.max()
    except: pass

    RESULTS[name] = dict(
        acc=accuracy_score(y_test_arr, y_pred),
        f1=f1_score(y_test_arr, y_pred, average='macro'),
        f1_M=f1_score(y_test_arr, y_pred, pos_label=1),
        f1_F=f1_score(y_test_arr, y_pred, pos_label=0),
        auc=roc_auc_score(y_test_arr, y_prob),
        prec=precision_score(y_test_arr, y_pred, average='macro'),
        rec=recall_score(y_test_arr, y_pred, average='macro'),
        mcc=matthews_corrcoef(y_test_arr, y_pred),
        ap=average_precision_score(y_test_arr, y_prob),
        y_prob=y_prob, y_pred=y_pred,
        t_train=t_train, t_inf_ms=t_inf
    )
    r = RESULTS[name]
    print(f"{name:<32} {r['acc']:>7.4f} {r['f1']:>7.4f} {r['auc']:>7.4f} {r['mcc']:>7.4f}  {t_train:.1f}s")
print("─"*70)
print("✅ 8 modèles entraînés")


## 4. Augmentation des données (pour le modèle DL)

In [ ]:
def char_noise(word, rng):
    if len(word) < 3: return word
    i = int(rng.integers(0, len(word)-1))
    chars = list(word)
    chars[i], chars[i+1] = chars[i+1], chars[i]
    return ''.join(chars)

def augmenter_nom(nom_parts, prenoms_parts, label, rng):
    rows = []
    all_parts = nom_parts + prenoms_parts
    for _ in range(N_AUGMENTATIONS):
        parts = all_parts.copy()
        rng.shuffle(parts)                                   # A. Permutation
        if prenoms_parts and rng.random() < PRENOMS_FIRST:  # B. Inversion
            p_s = prenoms_parts.copy(); rng.shuffle(p_s)
            n_s = nom_parts.copy();     rng.shuffle(n_s)
            parts = p_s + n_s
        if len(parts)>1 and rng.random() < WORD_DROP_PROB:  # C. Word dropout
            idx = int(rng.integers(0, len(parts)))
            parts = [p for i,p in enumerate(parts) if i!=idx]
        if parts and rng.random() < CHAR_NOISE_PROB:         # D. Char noise
            idx = int(rng.integers(0, len(parts)))
            parts[idx] = char_noise(parts[idx], rng)
        if parts:
            rows.append({'full_name':' '.join(parts), 'label':label})
    return rows

# Reconstruction du DataFrame d'entraînement original avec colonnes nom/prenoms
df_train_src = df.loc[X_train.index]
rng = np.random.default_rng(42)
aug_rows = []
for _, row in df_train_src.iterrows():
    aug_rows.append({'full_name': row['full_name'], 'label': row['label']})
    aug_rows.extend(augmenter_nom(row['nom_clean'].split(),
                                  row['prenoms_clean'].split(),
                                  row['label'], rng))

df_train_aug = pd.DataFrame(aug_rows).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Train original   : {len(df_train_src):,}")
print(f"Train augmenté   : {len(df_train_aug):,}  (×{len(df_train_aug)/len(df_train_src):.1f})")


## 5. Vocabulaire et encodage caractère

In [ ]:
# Construction du vocabulaire sur tout le corpus
all_texts = pd.concat([df_train_aug['full_name'], X_val, X_test])
chars = sorted(set(''.join(all_texts)))
char_to_index = {c: i+2 for i,c in enumerate(chars)}
char_to_index['<PAD>'] = 0
char_to_index['<UNK>'] = 1
vocab_size = len(char_to_index)
max_len = int(all_texts.str.len().quantile(0.99))

print(f"vocab_size = {vocab_size}  |  max_len = {max_len}")
print(f"Alphabet   : {sorted(chars)}")

def text_to_seq(text):
    return [char_to_index.get(c, 1) for c in str(text)]

def encode(texts, labels=None):
    seqs = [text_to_seq(t) for t in texts]
    X = pad_sequences(seqs, maxlen=max_len, padding='post', value=0).astype(np.int32)
    if labels is not None:
        return X, np.array(labels).astype(np.float32)
    return X

X_train_dl, y_train_dl = encode(df_train_aug['full_name'], df_train_aug['label'])
X_val_dl,   y_val_dl   = encode(X_val, y_val)
X_test_dl,  y_test_dl  = encode(X_test, y_test)

cw = compute_class_weight('balanced', classes=np.array([0,1]), y=y_train_dl)
cw_dict = {0:float(cw[0]), 1:float(cw[1])}
print(f"class_weights = {cw_dict}")
print(f"X_train shape = {X_train_dl.shape} | dtype = {X_train_dl.dtype}")


## 6. Architecture CNN + BiLSTM + MultiHeadAttention v4

In [ ]:
def build_model(vocab_size, max_len):
    """
    Architecture CNN_BiLSTM_MHA (identique à train_model.py)
    Embedding → SpatialDropout
    → Conv1D(k=3) + Conv1D(k=5) → Add  [trigrammes + quingrammes]
    → BiLSTM(128, return_seq=True)
    → MultiHeadAttention(4 têtes) + résidu + LayerNorm
    → BiLSTM(64)
    → Dense(128) → Dropout → Dense(64) → Dropout → sigmoid
    """
    inp = Input(shape=(max_len,), dtype=tf.int32, name='input')
    x   = Embedding(vocab_size, EMBED_DIM, name='embedding')(inp)
    x   = SpatialDropout1D(0.2)(x)
    c3  = Conv1D(CNN_FILTERS, 3, padding='same', activation='relu', name='conv3')(x)
    c5  = Conv1D(CNN_FILTERS, 5, padding='same', activation='relu', name='conv5')(x)
    x   = Add(name='cnn_merge')([c3, c5])
    x   = Bidirectional(LSTM(LSTM1, return_sequences=True,  dropout=0.2), name='bilstm1')(x)
    attn= MultiHeadAttention(MHA_HEADS, MHA_KEY_DIM, dropout=0.1, name='mha')(x, x)
    x   = LayerNormalization(name='layernorm')(Add()([x, attn]))
    x   = Bidirectional(LSTM(LSTM2, return_sequences=False, dropout=0.2), name='bilstm2')(x)
    x   = Dense(128, 'relu', name='dense1')(x); x = Dropout(0.4)(x)
    x   = Dense(64,  'relu', name='dense2')(x); x = Dropout(0.3)(x)
    out = Dense(1, 'sigmoid', name='output')(x)
    model = Model(inp, out, name='CNN_BiLSTM_MHA')
    lr = tf.keras.optimizers.schedules.CosineDecayRestarts(1e-3, 1000, t_mul=2.0, m_mul=0.9, alpha=1e-6)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTH),
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return model

model = build_model(vocab_size, max_len)
model.summary()


## 7. Entraînement du CNN + BiLSTM + MHA


In [ ]:
callbacks = [
    EarlyStopping(monitor='val_auc', mode='max', patience=7,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_gender_model.keras', monitor='val_auc', mode='max',
                    save_best_only=True, verbose=0),
]

print("▶ Entraînement en cours...")
t0 = time.time()
history = model.fit(
    X_train_dl, y_train_dl,
    validation_data=(X_val_dl, y_val_dl),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=cw_dict,
    callbacks=callbacks,
    verbose=1
)
t_train = time.time() - t0
print(f"\n Entraînement terminé en {t_train/60:.1f} min")

# Sauvegarde artefacts
with open('char_to_index.pkl','wb') as f: pickle.dump(char_to_index, f)
with open('max_len.pkl','wb') as f: pickle.dump(max_len, f)
print("✅ Artefacts sauvegardés : best_gender_model.keras | char_to_index.pkl | max_len.pkl")


## 7b. Export ONNX - livrable de production

In [ ]:
# ── Installation de tf2onnx si besoin ───────────────────────────────
import subprocess, sys
try:
    import tf2onnx
    import onnxruntime as ort
    print(" tf2onnx et onnxruntime déjà installés")
except ImportError:
    print("Installation de tf2onnx et onnxruntime...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'tf2onnx', 'onnxruntime', '-q'], check=True)
    import tf2onnx
    import onnxruntime as ort
    print(" Installation OK")

import tf2onnx, onnx, onnxruntime as ort
import numpy as np

ONNX_PATH = 'best_gender_model.onnx'

# ── Export ONNX (opset 15 - requis pour BiLSTM + MHA) ────────────────
print("\n▶ Export ONNX en cours...")
input_signature = [
    tf.TensorSpec(shape=[None, max_len], dtype=tf.int32, name='input')
]
onnx_model, _ = tf2onnx.convert.from_keras(
    model,
    input_signature=input_signature,
    opset=15,
    output_path=ONNX_PATH
)
print(f"✅ Modèle ONNX sauvegardé → {ONNX_PATH}")

# ── Vérification : prédictions ONNX identiques à Keras ───────────────
print("\n▶ Vérification ONNX vs Keras...")
sess    = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
in_name = sess.get_inputs()[0].name

sample      = X_test_dl[:8].astype(np.int32)
ort_out     = sess.run(None, {in_name: sample})[0].flatten()
keras_out   = model.predict(sample, verbose=0).flatten()
max_diff    = float(np.abs(ort_out - keras_out).max())

print(f"  Entrée   : {in_name}  shape={sample.shape}  dtype={sample.dtype}")
print(f"  Diff max ONNX vs Keras : {max_diff:.2e}  {'✅ OK (< 1e-4)' if max_diff < 1e-4 else '⚠️  > 1e-4'}")

# ── Test d'inférence ONNX sur quelques noms ───────────────────────────
print("\n▶ Test d'inférence ONNX sur des noms réels :")
noms_test = [
    "kouassi adjoua marie",
    "koffi jean marc",
    "coulibaly fatoumata",
    "ouattara ibrahim",
    "konan aya rosalie",
    "yao serge yannick axel",
]
print(f"  {'Nom':<35} {'Keras':>8}  {'ONNX':>8}  Décision")
print("  " + "─"*60)
for nom in noms_test:
    seq  = pad_sequences([text_to_seq(nom)], maxlen=max_len, padding='post', value=0).astype(np.int32)
    p_k  = float(model.predict(seq, verbose=0)[0][0])
    p_o  = float(sess.run(None, {in_name: seq})[0][0][0])
    genre = 'Masculin (M)' if p_o > 0.5 else 'Féminin  (F)'
    print(f"  {nom:<35} {p_k:>8.4f}  {p_o:>8.4f}  → {genre}")

# ── Résumé des artefacts produits ────────────────────────────────────
print("\n" + "="*55)
print("  ARTEFACTS PRODUITS")
print("="*55)
import os
artefacts = {
    'best_gender_model.onnx' : 'Modèle de production (ONNX Runtime)',
    'best_gender_model.keras': 'Sauvegarde Keras (réentraînement)',
    'char_to_index.pkl'      : 'Vocabulaire caractère (29 tokens)',
    'max_len.pkl'            : 'Longueur max de séquence (37)',
}
for fname, desc in artefacts.items():
    exists = os.path.exists(fname)
    size   = f"{os.path.getsize(fname)/1024:.0f} Ko" if exists else "-"
    status = "✅" if exists else "❌"
    print(f"  {status}  {fname:<30} {size:>8}   {desc}")
print("="*55)


## 8. Courbes d'apprentissage

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Courbes d'apprentissage - CNN + BiLSTM + MHA v4",
             fontsize=13, fontweight='bold', color=BLUE)

for ax, m, title, ylbl in zip(axes,
        ['loss','accuracy','auc'],
        ['Loss (label smoothed)','Accuracy','AUC-ROC'],
        ['Loss','Accuracy','AUC']):
    ax.plot(history.history[m],          color=LBLUE, lw=2,   label='Train')
    ax.plot(history.history[f'val_{m}'], color=ORANGE, lw=2, ls='--', label='Validation')
    ax.set_title(title, color=BLUE)
    ax.set_xlabel('Époque'); ax.set_ylabel(ylbl)
    ax.legend(fontsize=10); ax.grid(alpha=0.3)
    # Marquer le meilleur epoch
    best_e = int(np.argmax(history.history['val_auc'])) if m=='auc' else int(np.argmin(history.history['val_loss']))
    best_v = history.history[f'val_{m}'][best_e]
    ax.axvline(best_e, color='green', ls=':', lw=1.5, alpha=0.7)
    ax.scatter([best_e], [best_v], color='green', zorder=5, s=50)

plt.tight_layout()
plt.savefig('figures/fig5_courbes_apprentissage.png', dpi=150, bbox_inches='tight')
plt.show()
best_ep = int(np.argmax(history.history['val_auc']))
print(f"Meilleur epoch : {best_ep+1}/{len(history.history['val_auc'])}")
print(f"  val_accuracy = {history.history['val_accuracy'][best_ep]:.4f}")
print(f"  val_auc      = {history.history['val_auc'][best_ep]:.4f}")
print(f"  val_loss     = {history.history['val_loss'][best_ep]:.4f}")


## 9. Évaluation du modèle DL sur le set de test

In [ ]:
# Prédiction sur le test set
t1 = time.time()
y_prob_dl = model.predict(X_test_dl, verbose=0).flatten()
t_inf_dl  = (time.time()-t1)/len(X_test_dl)*1000
y_pred_dl = (y_prob_dl > 0.5).astype(int)

RESULTS['CNN + BiLSTM + MHA v4'] = dict(
    acc=accuracy_score(y_test_arr, y_pred_dl),
    f1=f1_score(y_test_arr, y_pred_dl, average='macro'),
    f1_M=f1_score(y_test_arr, y_pred_dl, pos_label=1),
    f1_F=f1_score(y_test_arr, y_pred_dl, pos_label=0),
    auc=roc_auc_score(y_test_arr, y_prob_dl),
    prec=precision_score(y_test_arr, y_pred_dl, average='macro'),
    rec=recall_score(y_test_arr, y_pred_dl, average='macro'),
    mcc=matthews_corrcoef(y_test_arr, y_pred_dl),
    ap=average_precision_score(y_test_arr, y_prob_dl),
    y_prob=y_prob_dl, y_pred=y_pred_dl,
    t_train=t_train, t_inf_ms=t_inf_dl
)

r = RESULTS['CNN + BiLSTM + MHA v4']
print("="*60)
print("  CNN + BiLSTM + MHA v4 - RÉSULTATS SET DE TEST")
print("="*60)
for lbl, val in [('Accuracy',r['acc']),('F1 macro',r['f1']),('AUC-ROC',r['auc']),
                  ('MCC',r['mcc']),('Précision',r['prec']),('Rappel',r['rec']),('Avg Prec.',r['ap'])]:
    print(f"  {lbl:<18}: {val:.4f}")
print(f"  {'Latence':<18}: {r['t_inf_ms']:.3f} ms/exemple")
print("="*60)
print()
print(classification_report(y_test_arr, y_pred_dl, target_names=['Féminin','Masculin'], digits=4))


In [ ]:
# Test d'invariance NOM/PRÉNOMS (comme dans train_model.py)
print("Test d'invariance à l'ordre NOM ↔ PRÉNOMS :")
print("-"*55)
test_pairs = [
    ("kouassi jean marie",   "jean marie kouassi"),
    ("akissi grace nadege",  "grace nadege akissi"),
    ("brou pascal ange",     "pascal ange brou"),
    ("coulibaly fatoumata",  "fatoumata coulibaly"),
    ("kone seydou abdoulaye","abdoulaye seydou kone"),
]
for nom_first, pre_first in test_pairs:
    preds = []
    for txt in [nom_first, pre_first]:
        seq  = pad_sequences([text_to_seq(txt)], maxlen=max_len, padding='post', value=0).astype(np.int32)
        prob = float(model.predict(seq, verbose=0)[0][0])
        preds.append(('M' if prob>0.5 else 'F', prob))
    inv = "✅ invariant" if preds[0][0]==preds[1][0] else "⚠️  différent"
    print(f"  {nom_first:<30} → {preds[0][0]} ({preds[0][1]:.3f})")
    print(f"  {pre_first:<30} → {preds[1][0]} ({preds[1][1]:.3f})  {inv}")
    print()


## 10. Tableau comparatif - toutes les métriques

In [ ]:
import pandas as pd

rows = []
for name, r in RESULTS.items():
    t_tr = f"{r['t_train']/60:.1f} min" if not np.isnan(r['t_train']) else "-"
    rows.append({
        'Modèle': name,
        'Accuracy': f"{r['acc']:.4f}",
        'F1 macro': f"{r['f1']:.4f}",
        'AUC-ROC':  f"{r['auc']:.4f}",
        'MCC':      f"{r['mcc']:.4f}",
        'Précision':f"{r['prec']:.4f}",
        'Rappel':   f"{r['rec']:.4f}",
        'F1 (M)':   f"{r['f1_M']:.4f}",
        'F1 (F)':   f"{r['f1_F']:.4f}",
        'Avg Prec.':f"{r['ap']:.4f}",
        'Latence (ms)': f"{r['t_inf_ms']:.3f}",
        'Entraîn.': t_tr,
    })

res_df = pd.DataFrame(rows)
res_df_num = res_df.set_index('Modèle').apply(pd.to_numeric, errors='coerce')
res_df = res_df.set_index('Modèle').loc[res_df_num['F1 macro'].sort_values(ascending=False).index]

print("TABLEAU COMPARATIF COMPLET - SET DE TEST (N={})".format(len(y_test_arr)))
print("="*100)
print(res_df.to_string())
print("="*100)
print("★ = meilleur modèle (CNN+BiLSTM+MHA v4)")


## 11. Visualisations finales - pour le mémoire

In [ ]:
# ── Fig 6 : Barres toutes métriques ─────────────────────────────────
names = list(RESULTS.keys())
colors = [MODEL_COLORS.get(n, GRAY) for n in names]

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle("Comparaison des modèles - métriques sur le set de test",
             fontsize=14, fontweight='bold', color=BLUE, y=1.01)

for ax, metric, label in zip(axes.flat, ['acc','f1','auc','mcc'],
                              ['Accuracy','F1 Macro','AUC-ROC','MCC']):
    vals = [RESULTS[n][metric] for n in names]
    si   = np.argsort(vals)
    sn   = [names[i] for i in si]
    sv   = [vals[i] for i in si]
    sc   = [colors[i] for i in si]
    bars = ax.barh(sn, sv, color=sc, edgecolor='white', linewidth=1.2)
    for b,v,n in zip(bars,sv,sn):
        ax.text(b.get_width()+0.002, b.get_y()+b.get_height()/2,
                f'{v:.4f}', va='center', fontsize=9,
                fontweight='bold' if 'CNN' in n else 'normal')
    ax.set_title(label, fontsize=13, color=BLUE, fontweight='bold')
    ax.set_xlim(min(sv)*0.97, 1.015)
    for b,n in zip(bars,sn):
        if 'CNN' in n: b.set_edgecolor(ORANGE); b.set_linewidth(2.5)
plt.tight_layout()
plt.savefig('figures/fig6_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Fig 7 : Courbes ROC ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
ax.plot([0,1],[0,1],'--', color='gray', lw=1.2, label='Aléatoire (AUC=0.50)')
for name, r in sorted(RESULTS.items(), key=lambda x: x[1]['auc'], reverse=True):
    fpr, tpr, _ = roc_curve(y_test_arr, r['y_prob'])
    lw = 2.5 if 'CNN' in name else 1.5
    ax.plot(fpr, tpr, lw=lw, color=MODEL_COLORS.get(name, GRAY),
            label=f"{name} (AUC={r['auc']:.4f})")
ax.set_xlabel('Taux Faux Positifs (FPR)', fontsize=12)
ax.set_ylabel('Taux Vrais Positifs (TPR)', fontsize=12)
ax.set_title('Courbes ROC - tous les modèles', fontsize=14, fontweight='bold', color=BLUE)
ax.legend(loc='lower right', fontsize=9, framealpha=0.9)
ax.set_xlim(0,1); ax.set_ylim(0,1.02)
plt.tight_layout()
plt.savefig('figures/fig7_roc.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Fig 8 : Matrices de confusion top 4 ─────────────────────────────
top4 = sorted(RESULTS.items(), key=lambda x: x[1]['f1'], reverse=True)[:4]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle("Matrices de confusion - Top 4 modèles", fontsize=14, fontweight='bold', color=BLUE)

for ax, (name, r) in zip(axes.flat, top4):
    cm = confusion_matrix(y_test_arr, r['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Féminin', 'Masculin'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')

    # Supprimer les cadres
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(left=False, bottom=False)

    ax.set_title(f"{name}\nAcc={r['acc']:.3f} | F1={r['f1']:.3f} | AUC={r['auc']:.3f}",
                 fontsize=10,
                 color=ORANGE if 'CNN' in name else BLUE,
                 fontweight='bold' if 'CNN' in name else 'normal')

    for i in range(2):
        tot = cm[i].sum()
        for j in range(2):
            ax.texts[i*2+j].set_text(f"{cm[i,j]}\n({cm[i,j]/tot*100:.1f}%)")

plt.tight_layout()
plt.savefig('figures/fig8_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 9 : Radar chart ──────────────────────────────────────────────
top6 = sorted(RESULTS.keys(), key=lambda n: RESULTS[n]['f1'], reverse=True)[:6]
metrics_r = ['acc','f1','auc','prec','rec','mcc']
labels_r  = ['Accuracy','F1','AUC-ROC','Précision','Rappel','MCC']
angles = np.linspace(0, 2*np.pi, len(metrics_r), endpoint=False).tolist() + [0]

fig, ax = plt.subplots(figsize=(10, 9), subplot_kw=dict(polar=True))
for name in top6:
    vals = [RESULTS[name][m] for m in metrics_r] + [RESULTS[name][metrics_r[0]]]
    lw = 2.5 if 'CNN' in name else 1.5
    ax.plot(angles, vals, lw=lw, color=MODEL_COLORS.get(name, GRAY), label=name)
    ax.fill(angles, vals, alpha=0.06, color=MODEL_COLORS.get(name, GRAY))
ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels_r, fontsize=11)
ax.set_ylim(0.75, 1.0)
ax.set_title("Profil multi-métriques - Top 6 modèles", fontsize=13, fontweight='bold', color=BLUE, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.15), fontsize=9.5)
ax.grid(color='gray', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/fig9_radar.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Fig 10 : Courbes Précision-Rappel ───────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
ax.axhline(y_test_arr.mean(), ls='--', color='gray', lw=1.2, label=f'Baseline ({y_test_arr.mean():.2f})')
for name, r in sorted(RESULTS.items(), key=lambda x: x[1]['ap'], reverse=True):
    prec, rec, _ = precision_recall_curve(y_test_arr, r['y_prob'])
    lw = 2.5 if 'CNN' in name else 1.5
    ax.plot(rec, prec, lw=lw, color=MODEL_COLORS.get(name, GRAY),
            label=f"{name} (AP={r['ap']:.4f})")
ax.set_xlabel('Rappel', fontsize=12); ax.set_ylabel('Précision', fontsize=12)
ax.set_title('Courbes Précision-Rappel - tous modèles', fontsize=13, fontweight='bold', color=BLUE)
ax.legend(loc='lower left', fontsize=9, framealpha=0.9)
ax.set_xlim(0,1); ax.set_ylim(0,1.02)
plt.tight_layout()
plt.savefig('figures/fig10_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 12. Synthèse finale

In [ ]:
print("="*70)
print("  SYNTHÈSE FINALE - RECONNAISSANCE DU GENRE")
print(f"  Corpus : {len(df):,} noms ivoiriens")
print(f"  Test   : {len(y_test_arr):,} exemples (15%, stratifié)")
print("="*70)

cols = ['acc','f1','auc','mcc','prec','rec','t_inf_ms']
hdrs = ['Acc','F1','AUC','MCC','Prec','Rec','ms/ex']
print(f"{'Modèle':<33} " + "  ".join(f"{h:>7}" for h in hdrs))
print("─"*88)

for name in sorted(RESULTS.keys(), key=lambda n: RESULTS[n]['f1'], reverse=True):
    r = RESULTS[name]
    star = " ★" if 'CNN' in name else ""
    vals = "  ".join(f"{r[c]:>7.4f}" for c in cols)
    print(f"{name+star:<33} {vals}")

print("="*70)
print("★ = Modèle proposé dans le mémoire")
print()
print("FIGURES PRODUITES :")
for f in sorted(os.listdir('figures')):
    print(f"  📊 figures/{f}")
